# pyads — Interactive Demo

This notebook walks through the core components of the **pyads** pipeline without
requiring a live Mistral API key or network connection. All API calls are mocked.

## Pipeline overview

```
PDF files
   └─► OCR (Mistral OCR API)          → data/text/*.txt
          └─► Extraction (Mistral LLM) → data/extracted/adsorption_data.json
                 └─► CIF download (COD) → cif_file/*.cif
                        └─► CIF analysis (gemmi + pymatgen) → cif_analysis_report.csv
```

In [ ]:
import json
import sys
from pathlib import Path
from unittest.mock import MagicMock

# Make sure the package is importable when running from examples/
sys.path.insert(0, str(Path('.').resolve().parent))

## 1. Extraction schema

Every extracted record follows a versioned schema. The fields below are guaranteed
to be present; any unextracted value is `null` (never guessed).

In [ ]:
from pyads.extractor import _empty_record

empty = _empty_record("example.txt")
print(json.dumps(empty, indent=2))

## 2. Extraction from OCR text (mocked LLM)

The extractor sends OCR text to Mistral and parses the response into the schema.
Here we mock the API call with a pre-written JSON response.

In [ ]:
from pyads.extractor import extract_data_from_text

OCR_TEXT = """
BET surface area: 1200 m2/g.  Pore volume: 0.55 cm3/g.  Pore size: 11.6 A.
CO2 adsorption isotherms at 273 K and 298 K.  N2 at 77 K.
Material: ZIF-8.  DOI: 10.1039/d0ta99999a.  Year: 2023.
"""

LLM_RESPONSE = json.dumps({
    "doi": "10.1039/d0ta99999a",
    "title": "CO2 and N2 adsorption on ZIF-8",
    "year": 2023,
    "material": "ZIF-8",
    "surface_area": {"value": 1200.0, "unit": "m2/g"},
    "pore_volume": {"value": 0.55, "unit": "cm3/g"},
    "pore_size": {"value": 11.6, "unit": "Angstrom"},
    "gases": ["CO2", "N2"],
    "isotherm_temperatures": [
        {"value": 273, "unit": "K"},
        {"value": 298, "unit": "K"},
        {"value": 77, "unit": "K"},
    ],
})

# Build a mock Mistral client
msg = MagicMock(); msg.content = LLM_RESPONSE
choice = MagicMock(); choice.message = msg
usage = MagicMock(); usage.model_dump.return_value = {"prompt_tokens": 312, "completion_tokens": 87, "total_tokens": 399}
response = MagicMock(); response.choices = [choice]; response.usage = usage
client = MagicMock(); client.chat.complete.return_value = response

record, token_usage = extract_data_from_text(OCR_TEXT, "zif8.txt", client, "mistral-small-latest")
print(json.dumps(record, indent=2, ensure_ascii=False))

## 3. Record normalisation rules

The normaliser enforces strict unit validation. Surface area must be in m²/g;
pore volume must be in cm³/g. Wrong units are silently set to `null`.

In [ ]:
from pyads.extractor import _normalize_record

# This should REJECT the surface_area because cm3/g is a pore-volume unit
bad_record = {
    "material": "MOF-5",
    "surface_area": {"value": 3000.0, "unit": "cm3/g"},   # wrong unit!
    "pore_volume": {"value": 1.55, "unit": "cm3/g"},
}
normalised = _normalize_record(bad_record, "mof5.txt")
print("surface_area after normalisation:", normalised["surface_area"])  # should be null
print("pore_volume after normalisation: ", normalised["pore_volume"])   # should be kept

## 4. Token usage and cost reporting

After a real run, usage is captured in `data/extracted/usage_summary.json`.
You can also surface an estimated cost by setting price env vars.

In [ ]:
print(f"Token usage: {token_usage}")
print(f"Total tokens: {token_usage['total_tokens']}")

# Rough cost estimate at Mistral Small pricing (~$0.20/Mtok input, $0.60/Mtok output)
input_cost  = token_usage["prompt_tokens"]     / 1_000_000 * 0.20
output_cost = token_usage["completion_tokens"] / 1_000_000 * 0.60
print(f"Estimated cost for this extraction: ${input_cost + output_cost:.6f}")

## 5. CIF material matching

After downloading CIFs from COD, each file is scored against the material name
using token overlap and text similarity.

In [ ]:
from pyads.cif_analyzer import material_match_score

# Simulate metadata extracted from a downloaded CIF
metadata = {
    "chemical_name": "ZIF-8",
    "chemical_formula": "C8 H10 N4 Zn",
    "publication_title": "Zeolitic imidazolate framework-8",
    "data_block": "zif-8",
}

result = material_match_score("ZIF-8", metadata, "ZnC8H10N4")
print(f"Match label : {result['match_label']}")
print(f"Match score : {result['match_score']}")
print(f"Reason      : {result['match_reason']}")

## 6. Running the full pipeline

Once you have a `.env` file with your `MISTRAL_API_KEY`, run:

```bash
# Full pipeline
pyads

# Skip OCR (text files already extracted)
pyads --skip-ocr

# Two-pass extraction with strict validation
pyads --skip-ocr --second-pass --skip-cif-download --skip-cif-analysis

# Dry run (no API calls)
pyads --dry-run
```

See `examples/extract_demo.py` for a pure-Python offline version.